In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import gwpy
import snr
from gwpy.timeseries import TimeSeries
from scipy.fft import fft, ifft

plt.rc('font',**{'family':'serif','serif':['Times']})
plt.rc('text', usetex=False)

/home/ale/miniconda3/envs/gwenv/lib/python3.10/site-packages/gwpy/time/__init__.py:36: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  from lal import LIGOTimeGPS


# This File will produce sine-gaussian signals and correlations from 2 streams of noise sets

In [2]:
def inner_product(a, b, s=None):
    """
    Detetermine the inner product of two vectors with optional weighting

    Parameters
    ----------
    a, b : array-like
       The vectors over which to determine the inner product
    s : array-like
       The weighting vector. Default is None.
    """
#  s)[2:-2]
    if s is not None:
        return ((np.sum(np.real(2 * ( a * np.conj(b)) / s))))
    else:
        return np.sum(np.real(2 * (a * np.conj(b))))


def snr(signal, noise):
    snr = np.sqrt(np.abs(inner_product(signal, signal, noise)))
    return snr


In [3]:
def MakeGaussianNoise(amplitude, noisetimes):
    np.random.seed(19680801)
    mu, sigma = 0,0.1
    s = np.random.normal(mu, sigma, 10000)
    
    s1 = s * amplitude
    #count, bins, ignored = plt.hist(s1,30, density=True)
    indexes = np.random.random(np.shape(noisetimes))
    indexes = (indexes*1000)
    inds = []

    for i in indexes:
        ii = int(i)
        inds.append(ii)

    GaussianNoise = []
    for i in inds:
        noise = s1[i]
        GaussianNoise.append(noise)
    
    return GaussianNoise
#print((indexes))
    

In [4]:
def AddNoiseToSignal(signal, flag, noise): # this function returns the original signal with the noise added in, if flag is True signal added at every second point, if False signal added at every point
    signal_with_noise = []
    signal_with_noise.append(signal[0])
    for i in range(len(signal)):
        
        if flag == True:
            if i == 0:
                continue
            else:
                if (i%2) ==0:
                    s_w_n = signal[i]+noise[int[i/2]]
                else: 
                    s_w_n = signal[i]
                
                signal_with_noise.append(s_w_n)
        else:
            if i == 0:
                continue
            else:
                s_w_n = signal[i]+noise[i]
                signal_with_noise.append(s_w_n)
    
    return signal_with_noise

In [5]:
def plotsignals(times, sig_w_noise, sig, noise, title, title2):
    import matplotlib.pyplot as plt
    fig, (ax1, ax2)= plt.subplots(1,2, figsize = (15,5))

    fig.suptitle('Sine-Gaussian Wave Packets')
    ax1.scatter(times, (sig_w_noise), c='thistle', s=0.1)
    ax1.plot(times,sig_w_noise, c='lightskyblue', label = 'Signal with Noise', linewidth=1)
    ax1.set_ylabel('Amplitude')
    ax1.set_xlabel('Time (s)')
    ax1.set_title(title)

    ax2.scatter(times, (sig), c='lightskyblue', s=0.1)
    ax2.plot(times,sig, c='lightskyblue', label='Signal', linewidth=1)
    ax2.scatter(noisetimes, noise, c = 'thistle', s=1, label='Noise')
    ax2.set_ylabel('Amplitude')
    ax2.set_xlabel('Time (s)')
    ax2.set_title(title2)

    ax1.set_facecolor('black')
    ax2.set_facecolor('black')

    ax1.legend(labelcolor='linecolor', facecolor='black')
    ax2.legend(labelcolor='linecolor', facecolor='black')



In [6]:
def plotcorrelations(times, correlated_sig, title):
    import matplotlib.pyplot as plt
    fig, (ax1)= plt.subplots(1,1, figsize = (7.5,5))

    fig.suptitle('Cross-Correlated Signals')
    ax1.scatter(times, correlated_sig, c='thistle', s=0.1)
    ax1.plot(times, correlated_sig, c='lightskyblue', label = 'Correlated Signal', linewidth=1)
    ax1.set_ylabel('Cross Correlation')
    ax1.set_xlabel('Time (s)')
    ax1.set_title(title)
    ax1.set_facecolor('black')
    ax1.legend(labelcolor='linecolor', facecolor='black')


In [7]:
def AddNoiseToSignal(signal, flag, noise): # this function returns the original signal with the noise added in, if flag is True signal added at every second point, if False signal added at every point
    signal_with_noise = []
   # signal_with_noise.append(signal[0])
    for i in range(len(signal)):
        
        if flag == True:
            if i == 0:
                s_w_n = noise[i]
            else:
                if (i%2) ==0:
                    s_w_n = signal[i]+noise[int[i/2]]
                else: 
                    s_w_n = signal[i]
                
                signal_with_noise.append(s_w_n)
        else:
          #  if i == 0:
            #    s_w_n = noise[i]
           # else:
            s_w_n = signal[i]+noise[i]
           # print(s_w_n)
            signal_with_noise.append(s_w_n)
          #  break
    
    return signal_with_noise

In [8]:
f_0SG = np.arange(3, 5000, 1)
QsSG = np.array([3,9,100])
# 12/12/24
# higher Q and lower frequency - CHANGE TO 3 Hz with Q = 3, or F0 = 100, Q=100

def SG_tau(Q, f0):
    t = Q/(np.sqrt(2)*np.pi*f0)
    return t


def SG_H_plus(Q, f0, t, alpha, amplitude, shift):
    tau = SG_tau(Q,f0)
    t0 = np.max(t)/2
    t0 = t0 + shift
    H = np.sin(2*np.pi*f0*(t0-t)) * np.exp(-((t0-t)**2/tau**2)) 
    h = (((1+alpha**2)/2) * H)# + (alpha * H)

    h = h * amplitude
    return h


## 2 streams of Gaussian Noise 

### Stream 1

In [9]:
num_files = 100
noisetimes = np.arange(0,1,0.00048828125)
print(len(noisetimes))
np.random.seed(29680801)
mu, sigma = 0,1.0
s = np.random.normal(mu, sigma, 2048*10000)
print(mu-np.mean(s))
amplitude = 1
s1 = s *amplitude
print('Mean of noise sample',np.mean(s1))
#plt.plot(np.arange(0,10000,1), s1)
noise_array_stream1 = []
i=0
for n in np.arange(0, num_files, 1):
    noise = s1[i:i+2048]
   # GaussianNoise_stream1.append(noise)
    noise_array_stream1.append(noise)
    i+=2048
    

print(np.shape(noise_array_stream1))   

2048
-0.00011904831805355545
Mean of noise sample 0.00011904831805355545
(100, 2048)


In [10]:
np.random.seed(23982047) 
mu, sigma = 0,1# set sigma to 1 11.02.2025
s = np.random.normal(mu, sigma, 2048*10000)
amplitude = 1
s1 = s *amplitude
print('Mean of noise sample',np.mean(s1))
noise_array_stream2 = []
i=0
for n in np.arange(0, num_files, 1):
    noise = s1[i:i+2048]
   # GaussianNoise_stream1.append(noise)
    noise_array_stream2.append(noise)
    i+=2048
    
print(np.shape(noise_array_stream2))

Mean of noise sample -7.103928865113932e-05
(100, 2048)


In [11]:
def cross_correlation(data1, data2, times):
    correlations = np.zeros((0,0))
    #cc = np.zeros((np.shape(data1)))
    d1_mean = np.mean(data1)
    d2_mean = np.mean(data2)
    
    corr = np.correlate((data1-d1_mean), (data2-d2_mean), "same")
    correlations = np.append(correlations, corr/len(data1))
   # print(corr)
    nx = len(times) # set signal length time +- 100 samples at 1/2048
    nxs = np.arange(-nx//2, nx//2, 1)

    return correlations, nxs
                  

In [12]:
import math
import random

def cc(x, y):
    correlation = sum((a) * (b) for a, b in zip(x, y)) 
    return correlation

def cross_correlation(data1, data2, times):
    correlations = []
    lags = np.arange(-len(times)/2, len(times)/2, 1)
    nx = len(lags)
    nxs = np.arange(-nx//2, nx//2, 1) #(-nx//2, nx//2, 1)
    for lag in (lags):
         data1_shifted = np.roll(data1, int(lag))
         correlation = cc(data1_shifted, data2)
         correlation = correlation/len(times)
         correlations.append(correlation)
        # if lag == 0 or lag == np.max(lags): print('lag = ', lag, ', correlation = ', correlation); fig, ax1 = plt.subplots(1,1); ax1.plot(times, data1, c='red'); ax1.plot(times, data2, c='blue')
    #print('correlations = ', np.shape(correlations), ', max correlation = ', np.max(np.abs(correlations)))
    return correlations, nxs
        

In [13]:
np.random.seed(19680801)
times = noisetimes
shifts1 = np.random.random((len(times)))
shifts1 = shifts1*(4*10**-3)
shifts2 = np.random.random((len(times)))
shifts2 = shifts2*(4*10**-3)
signals1 = []
signals2 = []
print(np.max(shifts1)*0.4)
print(np.max(shifts2)*0.4)

print(len(times)/2)

0.0015999105958567285
0.0015995203521987557
1024.0


In [ ]:
## Add signals

signals1 = []
signals2 = []
labels1 = np.zeros((len(times),1))
labels2 = np.zeros((len(times),1))
labels1_fin = []
labels2_fin = []
maxccs = []

for i in np.arange(0,num_files,1):
    shift1 = shifts1[i]
    shift2 = shifts2[i]
    t0 = np.max(times)/2
    t0 = t0 + shift1
   # print(np.mean(noise_array_stream1))
    noise1 = noise_array_stream1[i] - np.mean(noise_array_stream1[i])
    noise2 = noise_array_stream2[i] - np.mean(noise_array_stream2[i])
  #  print(len(noise1))
  #  print(len(noise1), len(noise2))
    for a in np.arange(0,10, 0.001):
      SG1 = SG_H_plus(QsSG[2], 200, times, 0.8, a, 0)
      sig = np.fft.fft(SG1, n=len(times))
      #print(len(sig), len(noise1))
     # snr_sig1 = snr(np.array(sig), np.array(np.fft.fft(noise1,n=len(times))))
     # snr_sig2 = snr(np.array(sig), np.array(np.fft.fft(noise2, n=len(times))))
      snr_sig1 = snr(np.array(SG1), noise=None)  # np.array(noise1)
     # print(snr_sig1)
      if snr_sig1 < 15.1 and snr_sig1 > 14.9:
        for ind in range(len(SG1)):
          if np.abs(SG1[ind]) > 0.0001:
            labels1[ind] = 1
     #   print('a =', a)
      #  print(snr_sig1, snr_sig2)
        SG2=SG1
        SG1_fin = SG1
        SG2_fin = SG1
        break 
  
    print(len(SG1_fin), len(noise1))
   # tau1 = SG_tau(QsSG[2], 3500)
    ints_sig1 = np.where(labels1==1)[0]
    #ints_sig2 = np.where(labels2==1)[0]
   
    SG1_fin_sec = SG1_fin[(1024-int(len(ints_sig1)/2)-100):(1024+int(len(ints_sig1)/2)+100)]
    SG2_fin_sec = SG2_fin[(1024-int(len(ints_sig1)/2)-100):(1024+int(len(ints_sig1)/2)+100)]
    times_sig = np.arange(0, len(SG1_fin_sec)*1/2048, 1/2048)
    noise1_sec = noise1[(1024-int(len(ints_sig1)/2)-100):(1024+int(len(ints_sig1)/2)+100)]
    noise2_sec = noise2[(1024-int(len(ints_sig1)/2)-100):(1024+int(len(ints_sig1)/2)+100)]
 
    sg1 = AddNoiseToSignal(SG1_fin_sec, False, noise1_sec)
    sg2 = AddNoiseToSignal(SG2_fin_sec, False, noise2_sec)    
    sig1 = np.fft.fft(sg1, n=len(times_sig))/(1/2048)
    sig2 = np.fft.fft(sg2, n=len(times_sig))/(1/2048)
    n1 = np.fft.fft(noise1_sec, n=len(times_sig))/(1/2048)
    n2 = np.fft.fft(noise1_sec, n=len(times_sig))/(1/2048)
  #  plotcorrelations(1/times_sig, sig1,'SG1 GaussNoise1 Correlation in 2 Streams')
    noise1_sec1 = noise1[0:511]
    noise1_sec2 = noise1[511:1023]
    noise1_sec3 = noise1[1023:1535]
    noise1_sec4 = noise1[1535:2047]

    noise2_sec1 = noise2[0:511]
    noise2_sec2 = noise2[511:1023]
    noise2_sec3 = noise2[1023:1535]
    noise2_sec4 = noise2[1535:2047]
    #print(np.shape(times_sig))
    #print(np.shape(sg1))
    ccsig = cross_correlation(sg1, sg2, times_sig)
  #  plotcorrelations(ccsig[1], ccsig[0], 'SG1 GaussNoise1 Correlation in 2 Streams')
    ccnoise1 = cross_correlation(noise1_sec1, noise2_sec1, times[0:511])
    ccnoise2 = cross_correlation(noise1_sec2, noise2_sec2, times[511:1023])
    ccnoise3 = cross_correlation(noise1_sec3, noise2_sec3, times[1023:1535])
    ccnoise4 = cross_correlation(noise1_sec4, noise2_sec4, times[1535:2047])

    final_sig = np.concatenate((ccnoise1[0], ccnoise2[0], ccnoise3[0], ccsig[0], ccnoise4[0]))
    
   # snr_sig1 = snr(np.array(SG1_fin), noise=np.array(noise1)) #np.array(noise1))
   # snr_sig2 = snr(np.array(SG2_fin), noise=np.array(noise2)) #np.array(noise2))
    snr_sec1 = snr(np.array(SG1), noise=None) #np.array(n1))
    snr_sec2 = snr(np.array(SG2), noise=None) #np.array(n2))

  #  print('snrs = ', snr_sig1, snr_sig2, snr_sec1, snr_sec2, i)
   # plottimes = np.arange(0,len(final_sig)*(1/2048), (1/2048))
    plottimes = np.arange(0, len(final_sig), 1)
   # plottimes2 = np.arange(0, len(final_sig2), 1)
    sn1 = AddNoiseToSignal(SG1_fin, False, noise1)
    sn2 = AddNoiseToSignal(SG2_fin, False, noise2)
   # plotsignals(times, sn2, SG2_fin, noise2, 'sn2', 'Gaussian Noise Stream 2, SG2')
   # plotsignals(times, sn1, SG1_fin, noise1, 'sn1', 'Gaussian Noise Stream 1, SG1')
    signals1.append(sn1)
    signals2.append(sn2)

    maxcc = np.max(final_sig)
    maxccs.append(maxcc)
  ##  plotcorrelations(plottimes, final_sig, 'Correlated Concatenated Signal')
  #  plotcorrelations(ccsig[1], ccsig[0], 'Correlated Signal New Function')
   # plotcorrelations(plottimes2, final_sig2, 'Correlated Concatenated Signal - np.correlate')
   # plotcorrelations(ccsig2[1], ccsig2[0], 'Correlated Signal np.correlate')
    #plotcorrelations(ccnoise1[1], ccnoise1[0], 'Correlated Signal')
    np.save('../TranAD_MGB/data/SG_COR_LOWF_SNR15/test/'+str(i)+'_GAUSS_Anomaly_Gaussnoise_test.npy', final_sig)
    # labels
    labels = np.zeros((len(final_sig),1))
    labels[1535:1535+(len(ccsig[0]))] = 1
   # print(indexes_start, indexes_end)
    #fig, (ax1) = plt.subplots(1,1)
    #ax1.plot(plottimes, labels, c='red')
   # ax2.plot(nxl, labels, c='darkblue')
    
    np.save('../TranAD_MGB/data/SG_COR_LOWF_SNR15/labels/'+str(i)+'_GAUSS_Anomaly_Gaussnoise_labels.npy', labels)
    print('saved for ',i)
    # 43 76 1 2 
    
    
    

2048 2048
saved for  0
2048 2048
saved for  1
2048 2048
saved for  2
2048 2048
saved for  3
2048 2048
saved for  4
2048 2048
saved for  5
2048 2048
saved for  6
2048 2048
saved for  7
2048 2048
saved for  8
2048 2048
saved for  9
2048 2048
saved for  10
2048 2048
saved for  11
2048 2048
saved for  12
2048 2048
saved for  13
2048 2048
saved for  14
2048 2048
saved for  15
2048 2048
saved for  16
2048 2048
saved for  17
2048 2048
saved for  18
2048 2048
saved for  19
2048 2048
saved for  20
2048 2048
saved for  21
2048 2048
saved for  22
2048 2048
saved for  23
2048 2048
saved for  24
2048 2048
saved for  25
2048 2048
saved for  26
2048 2048
saved for  27
2048 2048
saved for  28
2048 2048
saved for  29
2048 2048
saved for  30
2048 2048
saved for  31
2048 2048
saved for  32
2048 2048
saved for  33
2048 2048
saved for  34
2048 2048
saved for  35
2048 2048
saved for  36
2048 2048
saved for  37
2048 2048
saved for  38
2048 2048
saved for  39
2048 2048
saved for  40
2048 2048
saved for  41
20